In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from ast import literal_eval
import warnings
warnings.filterwarnings('ignore')

# Sistemas Recomendadores: MovieLens Dataset

## 1. Load and Explore Data

In [2]:
df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


## 2. Simple Recommender: Weighted Rating

In [3]:
C = df['vote_average'].mean()
print(f"Media de votos: {C}")

m = df['vote_count'].quantile(0.90)
print(f"Percentil 90 de votos: {m}")

q_movies = df.copy().loc[df['vote_count'] >= m]
print(f"Películas que califican: {q_movies.shape[0]}")
print(f"Total de películas: {df.shape[0]}")

Media de votos: 5.618207215134184
Percentil 90 de votos: 160.0
Películas que califican: 4555
Total de películas: 45466


In [4]:
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

q_movies['score'] = q_movies.apply(weighted_rating, axis=1)
q_movies = q_movies.sort_values('score', ascending=False)

q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


## 3. Content-Based Recommender: TF-IDF Vectorization

In [5]:
tfidf = TfidfVectorizer(stop_words='english')
df['overview'] = df['overview'].fillna('')
tfidf_matrix = tfidf.fit_transform(df['overview'])

print(f"Forma de la matriz TF-IDF: {tfidf_matrix.shape}")
print(f"Palabras en el vocabulario: {len(tfidf.get_feature_names_out())}")
print(f"Primeras 10 palabras del vocabulario:")
print(tfidf.get_feature_names_out()[5000:5010])

Forma de la matriz TF-IDF: (45466, 75827)
Palabras en el vocabulario: 75827
Primeras 10 palabras del vocabulario:
['avails' 'avaks' 'avalanche' 'avalanches' 'avallone' 'avalon' 'avant'
 'avanthika' 'avanti' 'avaracious']


## 4. Cosine Similarity Score

In [ ]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
print(f"Forma de la matriz de similitud coseno: {cosine_sim.shape}")

## 5. Inverse Mapping and Recommendation Function

In [ ]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
print(indices[:10])

In [ ]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

print("Películas similares a 'The Dark Knight Rises':")
print(get_recommendations('The Dark Knight Rises'))
print("\nPelículas similares a 'The Godfather':")
print(get_recommendations('The Godfather'))

## 6. Improved Recommender: Load Additional Metadata

In [ ]:
credits = pd.read_csv('./data/credits.csv')
keywords = pd.read_csv('./data/keywords.csv')

df = df.drop([19730, 29503, 35587, 35803], errors='ignore')

keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
df['id'] = df['id'].astype('int')

df = df.merge(credits, on='id')
df = df.merge(keywords, on='id')

print(f"Nuevo shape del DataFrame: {df.shape}")

In [ ]:
features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(literal_eval)

print("Datos parseados correctamente")

## 7. Data Extraction Functions

In [ ]:
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        if len(names) > 3:
            names = names[:3]
        return names
    return []

df['director'] = df['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(get_list)

print(df[['title', 'cast', 'director', 'keywords', 'genres']].head(3))

## 8. Preprocessing and Metadata Soup

In [ ]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

features = ['cast', 'keywords', 'director', 'genres']
for feature in features:
    df[feature] = df[feature].apply(clean_data)

def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])

df['soup'] = df.apply(create_soup, axis=1)

print(df[['soup']].head(2))

## 9. CountVectorizer and Improved Recommendations

In [ ]:
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])

print(f"Forma de la matriz CountVectorizer: {count_matrix.shape}")

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

df = df.reset_index()
indices = pd.Series(df.index, index=df['title'])

print("Recomendaciones mejoradas para 'The Dark Knight Rises':")
print(get_recommendations('The Dark Knight Rises', cosine_sim2))

print("\nRecomendaciones mejoradas para 'The Godfather':")
print(get_recommendations('The Godfather', cosine_sim2))